In [ ]:
# Install dependencies (Colab/Kaggle single cell)k
!pip -q install tensorflow tensorflow-datasets transformers accelerate evaluate

# Imports & hardware check
import platform
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BertTokenizer, TFBertForSequenceClassification

print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices('GPU'))

# Load IMDB dataset and peek
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,
    with_info=True
)
print(ds_info)
for text, label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() else "Negative")
    print(text.numpy().decode()[:250], "...\n")

# Tokenizer and tf.data pipeline
MAX_LENGTH = 256
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer loaded:", tokenizer.name_or_path)

def encode_review(review_input):
    if isinstance(review_input, bytes):
        review_text = review_input.decode("utf-8")
    elif hasattr(review_input, "numpy"):
        review_text = review_input.numpy().decode("utf-8")
    else:
        review_text = str(review_input)
    return tokenizer.encode_plus(
        review_text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
    )

def tf_encode(text, label):
    ids, mask, type_ids = tf.py_function(
        func=lambda t: list(encode_review(t).values()),
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32]
    )
    ids.set_shape([MAX_LENGTH])
    mask.set_shape([MAX_LENGTH])
    type_ids.set_shape([MAX_LENGTH])
    features = {"input_ids": ids, "attention_mask": mask, "token_type_ids": type_ids}
    return features, label

def prepare_dataset(dataset):
    return (dataset
            .map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)
            .shuffle(2000)
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE))

train_ds = prepare_dataset(ds_train)
test_ds  = prepare_dataset(ds_test)

# Model, compile, and summary
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False
)
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]
model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()

# Train
EPOCHS = 2
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS
)
print("History keys:", list(history.history.keys()))

# Evaluate
eval_metrics = model.evaluate(test_ds, return_dict=True)
print(eval_metrics)

# Inference helper
id2label = {0: "Negative", 1: "Positive"}

def predict_sentiment(text: str):
    enc = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors="tf"
    )
    outputs = model(enc, training=False)
    logits = outputs.logits
    probs = tf.nn.softmax(logits, axis=-1)[0].numpy()
    pred = int(probs.argmax())
    label = id2label[pred]
    return label, float(probs.max())

custom_sentence = "The onboarding emails were confusing, but the agent fixed everything politely."
label, confidence = predict_sentiment(custom_sentence)
print(f"Prediction: {label} (confidence={confidence:.3f})")
